In [ ]:
using Pkg; Pkg.activate(".")
using CairoMakie, LinearAlgebra, LaTeXStrings, ForwardDiff, FFTW 
include("spectral_tools.jl")

### PDE Example 2: Transport Problem

$$
  u_t + {\bf v} \cdot \nabla u = 0 \qquad \text{(PBC)}
$$
where ${\bf v}$ is a constant vector determining the direction of transport.

In [ ]:
function plot_soln(t, x, U)
    surface(x, x, U; title = "t = $(round(t, digits=2))", 
            size = (400, 300), 
            color = :viridis, clims = (-0.3, 1.1), colorbar=false)
end

# ------------------------------------------
# Problem setup
N = 32 
D = 2 
dt = π/(6N) 
tmax = 32.0
v = [0.7, 0.3]
u0 = (x1, x2) ->  exp(- 10 * (2+cos(x1)+cos(x2))/2 )
#------------------------------------------

X1, X2 = xgrid(D, N)
x = X1[:,1]
K1, K2 = kgrid(D, N)

# directional derivative operator in Fourier space 
VD̂ = im*K1*v[1] + im*K2*v[2]

# initial condition, we also need one additional v in the past
# (this takes one step of the PDE backward in time)
U = u0.(X1, X2)
Uold = U + dt * real.( ifft( VD̂ .* fft(U) ) )

# time-stepping loop (check errors)
tt = 0:dt:tmax
errs = Float64[] 
nrmu2 = Float64[] 
@gif for t in tt 
    global U, Uold, VD̂, X1, X2, v, nrmu2
    # differentiation in reciprocal space
    W = real.( ifft( VD̂ .* fft(U) ) )
    # multiplication and update in real space
    U, Uold = Uold - 2 * dt * W, U
    # check error 
    Ue = u0.(X1 .- v[1]*t, X2 .- v[2]*t)
    push!(errs, norm(U[:] - Ue[:], Inf))
    push!(nrmu2, norm(U[:])^2 / (2*N)^2)
    # plot snapshot 
    plot_soln(t, x, U)
end every 10

In [ ]:
p1 = plot(tt, errs, lw=2, label = L"||u_N(t) - u(t)||_\infty")
plot!(tt, dt .+ dt^2 * tt, lw=2, ls=:dash, label = L"\Delta t + t \Delta t^2")
p2 = plot(tt, abs.(nrmu2 .- nrmu2[1]), label = L"\|U(t)\|_2^2")
plot(p1, p2, layout = (2,1))